In [1]:
import pandas as pd
from Bio import SeqIO
from Bio import SearchIO

In [2]:
df = pd.read_csv("tree_awr_candidate.tsv", sep = "\t")

In [3]:
with open("awr_candidate.fasta", "w") as f:
    for entry, seq in zip(df["entry"], df["seq"]):
        f.write(f">{entry}\n{seq}\n")

In [4]:
seq_db = {}

In [5]:
for record in SeqIO.parse("./awr_candidate.fasta", "fasta"):
    entry = record.id
    seq = str(record.seq)

    seq_db[entry] = seq

# Step 1. Initial guess and search on AWR candidate

In [6]:
with open("wy1.fasta", "w") as f:
    for record in SeqIO.parse("./awr_candidate.fasta", "fasta"):
        entry = record.id
        seq = str(record.seq)
    
        wy1 = seq[10:60]
        if len(wy1) != 50:
            continue
        
        f.write(f">{entry}_wy1\n{wy1}\n")

In [7]:
with open("wy2.fasta", "w") as f:
    for record in SeqIO.parse("./awr_candidate.fasta", "fasta"):
        entry = record.id
        seq = str(record.seq)
    
        wy2 = seq[70:120]

        if len(wy2) != 50:
            continue
        f.write(f">{entry}_wy2\n{wy2}\n")

In [8]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1.fasta > wy1.step1.align.fasta
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy2.fasta > wy2.step1.align.fasta

In [9]:
!conda run -n homology-env esl-reformat -o wy1.step1.align.sto stockholm wy1.step1.align.fasta
!conda run -n homology-env esl-reformat -o wy2.step1.align.sto stockholm wy2.step1.align.fasta

In [10]:
!conda run -n homology-env hmmbuild wy1.step1.hmm wy1.step1.align.sto
!conda run -n homology-env hmmbuild wy2.step1.hmm wy2.step1.align.sto

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             wy1.step1.align.sto
# output HMM file:                  wy1.step1.hmm
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     wy1.step1.align         83    98    50    12.74  1.106 

# CPU time: 0.08u 0.00s 00:00:00.08 Elapsed: 00:00:00.08

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - 

In [11]:
!cat wy1.step1.hmm wy2.step1.hmm > awr.step1.hmm

In [12]:
!conda run -n homology-env hmmsearch -o awr.step1.results awr.step1.hmm ./awr_candidate.fasta

# Step2. search on whole proteome profile

In [13]:
for i, query_result in enumerate(SearchIO.parse("awr.step1.results", "hmmer3-text")):
    i += 1
    print(query_result.id)
    with open(f"wy{i}.step2.fasta", "w") as f:
        for i, hit in enumerate(query_result.hits):
            print(hit.id)
            for j, hsp in enumerate(hit.hsps):
                if hsp.evalue < 1e-10:
                    
                    print(hsp.env_start, hsp.env_end, hsp.evalue)
                    print(seq_db[hit.id][hsp.env_start:hsp.env_end])
                    f.write(f">{hit.id}_{j}_wy{i}\n{seq_db[hit.id][hsp.env_start:hsp.env_end]}\n")

        print()

wy1.step1.align
Phylat_28083.11205
5 55 2.1e-25
TKWMRTTFWLNVGKSDEQVKKTLGLDRLSGAALKASPNYVYYEHFLYALE
125 175 4.6e-12
EKNAKVDIWISAERPSWYVRKVLDLDRRSTNAFWGSENYKYYEKFLYGIE
245 291 2.1e-12
EMKVKVEIWASANRPKEYVEKMLGLDKFALTASPNYSYYKDFLQLL
RAW31646
5 55 9.2e-25
TKWFRTTYWLNSGKTDEYVKKALGLDNLSGATLKSAPNYVYYEHFLDALE
125 175 1.5e-11
EMNAKVDMWISLKRPNWYVKKMLNLDHRSINAFRNSPNYPRYERFEDAMQ
243 289 2.2e-11
VNTKVEIWASKNRPNEYVEEILGLNGAADKTSANYKYYEYFLTLKK
POM69181
5 55 7.9e-25
LKSMRTSFWLETGKSDEYVKKTLELDNLSEAALKSAPNYQYYEHFLFTRE
241 287 1e-12
EMEVKVKIWASLKRPDQYVEELLNLDRNALKTSPNYEFYQKFLRGT
Phyra82793
6 56 1.7e-26
KNWGKMKYWLAAGKSDDQVKKALNMEGLTGAALKAHPNYKQLEKFWYKRE
126 176 4.7e-14
EMKVKVQIWANAKRDNGYVKEMLRLDGLSKAKREINPNNKYYLEFLKLTK
Phyinf1_7951
5 55 3e-22
IKRLRTATWIETGRTDDYVKTTLRLDNLSGAALKSAPNYMYYEHFLNALE
236 281 3e-11
ISTKVTIWASKDRPFEYVKKVLGLRGAADTTNANYKYFEDFLLQT
Phyra83943
6 56 2e-26
KNWGNMQYWLAVGKSDDQVKKALKMEGLTDAALKAHPNYKQLEKFWYKRE
126 175 7.9e-12
EMKVKVQIWAKAKRKSWYVKEMLDLDGLSKAMRETNPNNKYYLEFLKLT
XP_009537211.1
6 5

In [14]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1.step2.fasta > wy1.step2.align.fasta
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy2.step2.fasta > wy2.step2.align.fasta

In [15]:
!conda run -n homology-env esl-reformat -o wy1.step2.align.sto stockholm wy1.step2.align.fasta
!conda run -n homology-env esl-reformat -o wy2.step2.align.sto stockholm wy2.step2.align.fasta

In [16]:
!conda run -n homology-env hmmbuild wy1.step2.hmm wy1.step2.align.sto
!conda run -n homology-env hmmbuild wy2.step2.hmm wy2.step2.align.sto

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             wy1.step2.align.sto
# output HMM file:                  wy1.step2.hmm
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     wy1.step2.align         99    61    50    10.49  1.106 

# CPU time: 0.08u 0.00s 00:00:00.08 Elapsed: 00:00:00.10

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - 

In [17]:
!cat wy1.step2.hmm wy2.step2.hmm > awr.step2.hmm

In [18]:
!conda run -n homology-env hmmsearch -o awr.step2.results awr.step2.hmm ../all_protein.faa

# Step 3. Final build and search 

In [19]:
all_seq_db = {}
for record in SeqIO.parse("../all_protein.faa", "fasta"):
    entry = record.id
    seq = str(record.seq)

    all_seq_db[entry] = seq

In [20]:
for i, query_result in enumerate(SearchIO.parse("awr.step2.results", "hmmer3-text")):
    i += 1
    print(query_result.id)
    with open(f"wy{i}.step3.fasta", "w") as f:
        for i, hit in enumerate(query_result.hits):
            print(hit.id)
            for j, hsp in enumerate(hit.hsps):
                if hsp.evalue < 1e-10:
                    
                    print(hsp.env_start, hsp.env_end, hsp.evalue)
                    print(all_seq_db[hit.id][hsp.env_start:hsp.env_end])
                    f.write(f">{hit.id}_{j}_wy{i}\n{all_seq_db[hit.id][hsp.env_start:hsp.env_end]}\n")

        print()

wy1.step2.align
POM76046
529 578 9.4e-11
EMLTRAEIWGTAGRSKSYVKSMLGMDNVTSLKVKEHVNYRYYMKYRASR
POM71075
84 134 5.5e-11
ILNMRMKRWSKADKPAEYIQKALGLNKLSGTAFTSASNFRYYDAYMDDQV
POM63952
Phyci1_20366
0 49 5.1e-13
MKARVKIWSKNEKPLDFVKKQLGMEHLSEAAFLAAPNFKYYDDFVTSQR
298 348 8.3e-13
ELRMKTQIWTSAKRPEEYIKYSLGLKGLEGDALKAAPTYKFFEYYLDARK
Phyra76365
123 170 7.5e-12
TSQLPIWAKKELTPDEVMEKLGIEKLSGEALTSNPNFKYFDEFVSAQ
211 259 7.4e-11
VNQVRSWLKTDLSLSDAMVKLKLDKLSGDALLSHPNYGYYKSFVINKL
375 425 5.5e-12
ELQAKTLIWVEAKRPEWYVKYSLGLDDLEGEALKEAANFQFLERYLEGIK
OWZ15424
106 154 1.9e-12
PMESKLVVWSNNGKSVSFVKKELGLENLSDAAFKQAKNFKYYDDFVVS
405 455 1.2e-12
ELAQKTNIWISNKRPEWLVKLSLGLTGLEDDALKAHKNFQFYERYLDGMK
OWZ22789
109 156 2e-11
KSRLEAWSKSGKSVSFVKKELGLENLSGVAFKEAKNFQYYDDFVISQ
154 201 1.6e-11
SQLPIWTKKQLTPDEVIAQLGLQGLSGTALKTNPNFKYYEEFLKTQI
406 456 2.8e-11
ELWEKTLIWTSNKRPDWYVKYSLGLDGLGENAMKEAPNLKLYEYYLDGMK
OWZ06867
130 180 6.9e-12
MMKLKLHVWAENGKSATFVQKELGLNELWGAAFTGAANFKYYDDYVVSQL
429 478 2.5e-11
LQEKTFIWTSSKRPGWYVKYSLGLEGLGDEALKKSANYEF

In [21]:
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy1.step3.fasta > wy1.step3.align.fasta
!conda run -n homology-env mafft --thread 72 --maxiterate 1000 --quiet --globalpair wy2.step3.fasta > wy2.step3.align.fasta

In [22]:
!conda run -n homology-env esl-reformat -o wy1.step3.align.sto stockholm wy1.step3.align.fasta
!conda run -n homology-env esl-reformat -o wy2.step3.align.sto stockholm wy2.step3.align.fasta

In [23]:
!conda run -n homology-env hmmbuild wy1.final.hmm wy1.step3.align.sto
!conda run -n homology-env hmmbuild wy2.final.hmm wy2.step3.align.sto

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             wy1.step3.align.sto
# output HMM file:                  wy1.final.hmm
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     wy1.step3.align        233    62    50    11.02  1.106 

# CPU time: 0.08u 0.00s 00:00:00.08 Elapsed: 00:00:00.09

# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - 

In [24]:
!cat wy1.final.hmm wy2.final.hmm > awr.final.hmm

In [25]:
!conda run -n homology-env hmmsearch -o awr.final.results awr.final.hmm ../all_protein.faa